<a id="projectile-main"></a>
# 02-1. PhysicsNeMo 기본 — 발사체 운동 PINN

**세션:** 14:40–16:00 (PhysicsNeMo 기본 / PINN)  
**선행:** [GH200 GPU 메모리와 프로파일링](../01_GH200/02_GPU_Memory_Profile.ipynb)

**핵심 질문:** 정답 궤적을 학습 데이터로 쓰지 않고, 초기조건과 운동방정식만으로 `x(t), y(t)`를 근사할 수 있을까요?

이 실습에서는 물리 정보 신경망(Physics-Informed Neural Network, PINN)이 운동방정식을 얼마나 잘 만족하는지를 손실값으로 계산하는 방법과 PhysicsNeMo-Sym의 구성 요소를 배웁니다. 이 PINN은 고정된 초기조건에서 ODE 잔차를 줄이며 궤적을 근사합니다. 다음 FNO 실습은 여러 소스항·해의 쌍 `(f,u)`을 학습해 새로운 소스항 `f`에 대응하는 해 `u`를 예측합니다.

**완료 기준:** 학습 스크립트 실행 성공 → 검증 결과 파일 확인 → 해석적으로 구한 정답과 PINN 예측의 오차 계산 → 그래프 확인.


## 진행 순서

1. 환경과 지원 파일 확인
2. 해석해로 문제 설정 확인
3. PINN의 입력·출력·손실 이해
4. PhysicsNeMo-Sym의 구성 절차와 실제 코드 연결
5. Hydra 학습 설정 확인
6. 새 출력 폴더에서 학습을 시작하거나 지정한 체크포인트에서 이어서 학습
7. 해석해와 PINN 예측의 오차 분석

노트북에서는 핵심 구성을 단계별로 확인하고, 전체 학습은 [`../labs/projectile/source_code/projectile.py`](../labs/projectile/source_code/projectile.py)로 실행합니다.


## 1. 환경과 실습 폴더 확인

`00_Start_Here.ipynb`와 `01_GH200/02_GPU_Memory_Profile.ipynb`의 점검을 마쳤다는 전제로 발사체 운동 실습의 지원 폴더를 찾습니다.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import time

launch_dir = Path.cwd().resolve()
REPO_ROOT = next(
    (
        candidate
        for candidate in (launch_dir, *launch_dir.parents)
        if (candidate / "labs" / "projectile" / "source_code" / "projectile.py").is_file()
    ),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError(
        "labs/projectile/source_code/projectile.py를 찾지 못했습니다. "
        "KSC2026 과정 폴더 안에서 notebook을 열었는지 확인하세요."
    )

LAB_DIR = REPO_ROOT / "labs" / "projectile"
SOURCE_DIR = LAB_DIR / "source_code"
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

required = [
    SOURCE_DIR / "projectile.py",
    SOURCE_DIR / "projectile_eqn.py",
    SOURCE_DIR / "conf" / "config.yaml",
    LAB_DIR / "images" / "projectile.svg",
    LAB_DIR / "images" / "physicsnemo_sym_workflow.webp",
]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError(f"필수 파일이 없습니다: {missing}")

print(f"과정 루트       : {REPO_ROOT}")
print(f"발사체 실습 폴더: {LAB_DIR}")
print("필수 파일       : PASS")


<a id="projectile-problem"></a>
## 2. 문제: 초기속도로 쏜 물체의 위치

원점에서 초기속도 `v₀=40 m/s`, 발사각 `θ=60°`로 출발한 점질량의 위치 `(x(t), y(t))`를 구합니다. 중력가속도의 **크기**는 `g=9.81 m/s²`이고 위쪽을 양의 `y` 방향으로 두므로 `y''=-g`입니다. 공기저항과 지면 충돌은 모델에 포함하지 않습니다.

<p align="center"><img src="../labs/projectile/images/projectile.svg" width="620" alt="발사체 운동의 좌표축, 초기속도와 포물선 궤적" /></p>
<p align="center"><em>초기속도를 수평·수직 성분으로 나누면 중력은 수직 방향 속도만 바꿉니다.</em></p>

$$
x''(t)=0, \qquad y''(t)=-g
$$

$$
x(t)=v_0\cos\theta\,t, \qquad
y(t)=v_0\sin\theta\,t-\frac{1}{2}gt^2
$$

위 식으로 구한 해석해는 **검증 단계에서 정답으로만 사용**하며 PINN 학습에는 사용하지 않습니다.


In [ ]:
import numpy as np

initial_speed = 40.0             # m/s
launch_angle = np.deg2rad(60.0)  # rad
gravity = 9.81                   # m/s^2 (magnitude)

velocity_x = initial_speed * np.cos(launch_angle)
velocity_y = initial_speed * np.sin(launch_angle)
ground_return_time = 2.0 * velocity_y / gravity

def exact_position(t):
    t = np.asarray(t)
    x = velocity_x * t
    y = velocity_y * t - 0.5 * gravity * t**2
    return x, y

x_at_5, y_at_5 = exact_position(5.0)
print(f"초기 속도 성분 : vx={velocity_x:.2f} m/s, vy={velocity_y:.2f} m/s")
print(f"t=5 s의 위치   : x={x_at_5:.2f} m, y={y_at_5:.2f} m")
print(f"지면 복귀 시각 : t≈{ground_return_time:.2f} s (충돌은 모델링하지 않음)")


## 3. PINN은 무엇을 학습하나요?

```text
t ──► 완전연결 신경망 ──► x̂(t), ŷ(t)
                              │
                              └─ 자동미분 ─► x̂'', ŷ''
```

신경망은 시간 `t`를 받아 위치 `x̂, ŷ`를 출력합니다. 자동미분으로 시간에 대한 도함수를 계산하고 다음 두 손실의 합을 최소화합니다.

$$L=L_{IC}+L_{ODE}$$

- `L_IC`: `t=0`에서 `x=y=0`, `x'=v₀ cosθ`, `y'=v₀ sinθ`
- `L_ODE`: `r_x=x̂''=0`, `r_y=ŷ''+g=0`

두 식이 2계 ODE이므로 위치 두 개와 속도 두 개, 모두 네 개의 초기조건이 필요합니다. 학습 중에는 궤적의 정답 값을 사용하지 않고 초기조건과 운동방정식의 잔차를 사용합니다.


<a id="physicsnemo-workflow"></a>
## 4. PhysicsNeMo-Sym 학습 구성도

<p align="center"><img src="../labs/projectile/images/physicsnemo_sym_workflow.webp" width="900" alt="PhysicsNeMo-Sym에서 문제를 구성하고 학습을 실행하는 절차" /></p>

<p align="center"><em>PhysicsNeMo-Sym에서는 설정과 방정식·모델을 준비하고, 학습·검증·추론 구성 요소를 Domain에 등록한 뒤 Solver로 학습을 실행합니다.</em></p>

| 구성 단계 | 발사체 운동 코드 | 역할 |
|---|---|---|
| Load Hydra | `source_code/conf/config.yaml` | 신경망, 최적화 방법, 학습 단계 수 설정 |
| Define Geometry | `Point1D(0)` + 시간 변수 `t` | Constraint API용 고정점을 만들고, 시간 parameterization에서 실제 입력 `t` 샘플링 |
| Create Nodes | 방정식 노드 + 완전연결 신경망 | `t → x̂,ŷ` 계산과 자동미분을 이용한 잔차 계산 |
| Create Constraint | `initial_condition`, `ode_constraint` | 초기조건과 ODE 잔차를 학습 손실로 구성 |
| Create Validator | `0≤t<5` 구간의 해석해 | 학습 범위에서 정답과 예측 비교 |
| Create Inferencer | `0≤t<8` 구간의 시간 입력 | 정답 없이 예측 생성 |
| Create Monitor | 사용하지 않음 | 이 실습에서는 등록하지 않음 |
| Create Domain | `projectile_domain` | 학습·평가 구성 요소 등록 |
| Create/Run Solver | `Solver(...).solve()` | 등록된 구성으로 학습 실행 |

`N_c`, `N_v`, `N_i`, `N_m`은 Domain에 등록한 Constraint, Validator, Inferencer, Monitor의 개수를 나타냅니다.

출처: [NVIDIA PhysicsNeMo-Sym User Guide — PINNs Tutorials](https://docs.nvidia.com/physicsnemo/25.11/user-guide/pinns-tutorials/index.html)


<a id="step-2-equations-nodes"></a>
## 5. 방정식 노드와 신경망 노드

`ProjectileEquation`은 운동방정식을 잔차가 0이 되는 형태로 정의합니다.

```python
self.equations = {
    "ode_x": x.diff(t, 2),
    "ode_y": y.diff(t, 2) + gravity,
}
```

`instantiate_arch()`는 Hydra 설정에 따라 완전연결 신경망을 만들고, 입력 키 `t`와 출력 키 `x`, `y`를 연결합니다. 방정식 노드와 신경망 노드를 같은 목록에 넣으면 PhysicsNeMo가 자동미분을 포함한 계산 그래프를 구성합니다.


In [ ]:
from projectile_eqn import ProjectileEquation

equation = ProjectileEquation(gravity=gravity)
print("등록한 방정식 잔차:")
for name, expression in equation.equations.items():
    print(f"  {name}: {expression} = 0")


<a id="step-3-domain-constraints"></a>
## 6. 학습 조건(`Constraint`)과 검증·추론 구성

```python
projectile_domain.add_constraint(initial_condition, "initial_condition")
projectile_domain.add_constraint(ode_constraint, "ode_constraint")
projectile_domain.add_validator(validator)
projectile_domain.add_inferencer(grid_inference, "inferencer_data")
```

`PointwiseBoundaryConstraint`는 `t=0`에서 위치와 속도의 네 초기조건을 적용합니다. PhysicsNeMo-Sym의 Constraint API가 Geometry를 요구하므로 `Point1D(0)`을 고정점으로 사용하고, 신경망 입력 `t`는 시간 parameterization에서 샘플링합니다.

| 구성 요소 | 학습에 사용? | 정답 필요? | 용도 |
|---|---:|---:|---|
| 초기조건 `Constraint` | 예 | 초기값 필요 | 시작 위치와 속도 고정 |
| ODE `Constraint` | 예 | 정답 궤적 불필요 | 운동방정식의 잔차 최소화 |
| `Validator` | 아니오 | 해석해 필요 | `0–5 s` 구간의 정확도 평가 |
| `Inferencer` | 아니오 | 불필요 | `0–8 s` 구간의 예측 파일 생성 |


## 7. Hydra 설정을 실제 파일에서 확인

노트북에 설정값을 따로 옮겨 적으면 실제 실행 파일과 어긋날 수 있습니다. 아래 셀은 [`config.yaml`](../labs/projectile/source_code/conf/config.yaml)을 직접 읽습니다.


In [ ]:
import yaml

config_path = SOURCE_DIR / "conf" / "config.yaml"
config = yaml.safe_load(config_path.read_text(encoding="utf-8"))

print(f"설정 파일       : {config_path}")
print(f"최대 학습 단계  : {config['training']['max_steps']}")
print(f"검증 주기       : {config['training']['rec_validation_freq']}")
print(f"추론 주기       : {config['training']['rec_inference_freq']}")
print(f"초기조건 배치   : {config['batch_size']['initial_x']}")
print(f"ODE 내부점 배치 : {config['batch_size']['interior']}")
print(f"CUDA 그래프     : {config['cuda_graphs']}")


<a id="step-6-solver-training"></a>
## 8. Solver 실행: 새 학습 시작과 중단된 학습 재개

실제 스크립트는 `Solver(cfg, projectile_domain).solve()`를 호출합니다. 기본값은 실행 시각을 마이크로초 단위까지 이름에 넣은 **새 출력 폴더**이므로 이전 체크포인트나 검증 결과와 섞이지 않습니다.

- 처음 실행: `RESUME_RUN_DIR = None` 유지
- 중단된 실행 재개: 강사가 확인한 기존 실행 폴더의 절대 경로를 `RESUME_RUN_DIR`에 지정

재개할 폴더를 자동으로 고르지 않습니다. 잘못된 체크포인트를 불러오는 일을 막기 위해 참가자가 재개 대상을 직접 지정합니다.


In [ ]:
from datetime import datetime

RESUME_RUN_DIR = None  # 예: LAB_DIR / "outputs" / "ksc_projectile" / "20260827_144012_123456"

if RESUME_RUN_DIR is None:
    RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    RUN_OUTPUT_DIR = LAB_DIR / "outputs" / "ksc_projectile" / RUN_ID
    RUN_MODE = "새 학습"
else:
    RUN_OUTPUT_DIR = Path(RESUME_RUN_DIR).expanduser().resolve()
    if not RUN_OUTPUT_DIR.is_dir():
        raise FileNotFoundError(f"재개할 run 폴더가 없습니다: {RUN_OUTPUT_DIR}")
    RUN_MODE = "학습 재개"

command = [
    sys.executable,
    "source_code/projectile.py",
    f"network_dir={RUN_OUTPUT_DIR}",
]
print(f"실행 방식 : {RUN_MODE}")
print(f"결과 폴더 : {RUN_OUTPUT_DIR}")
print("실행 명령 :", " ".join(str(part) for part in command))

started = time.perf_counter()
completed = subprocess.run(command, cwd=LAB_DIR)
TRAIN_WALL_SECONDS = time.perf_counter() - started

print(f"전체 실행 시간: {TRAIN_WALL_SECONDS / 60:.2f} min")
if completed.returncode != 0:
    raise RuntimeError(f"Projectile 학습 실패 (exit code={completed.returncode})")

validator_path = RUN_OUTPUT_DIR / "validators" / "validator.npz"
if not validator_path.is_file():
    raise FileNotFoundError(f"검증 결과를 찾지 못했습니다: {validator_path}")
print(f"검증 결과: {validator_path}")


<a id="visualize-results"></a>
## 9. 현재 실행 결과를 직접 해석

아래 셀은 이번 실행에서 생성한 `validator.npz`를 읽어 해석해와 PINN 예측을 비교하고, 상대 L2 오차와 최대 절대 오차를 계산합니다.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

payload = np.load(validator_path, allow_pickle=True)
validation = np.atleast_1d(payload.f.arr_0)[0]

t = validation["t"][:, 0]
x_true, x_pred = validation["true_x"][:, 0], validation["pred_x"][:, 0]
y_true, y_pred = validation["true_y"][:, 0], validation["pred_y"][:, 0]

def relative_l2(prediction, target):
    return np.linalg.norm(prediction - target) / np.linalg.norm(target)

metrics = {
    "x_relative_l2": relative_l2(x_pred, x_true),
    "y_relative_l2": relative_l2(y_pred, y_true),
    "x_max_abs_error_m": np.max(np.abs(x_pred - x_true)),
    "y_max_abs_error_m": np.max(np.abs(y_pred - y_true)),
}
for name, value in metrics.items():
    print(f"{name:22s}: {value:.6e}")

figure, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
axes[0].plot(t, x_true, label="x(t) exact")
axes[0].plot(t, x_pred, "--", label="x(t) PINN")
axes[0].set(title="x(t)", xlabel="t (s)", ylabel="x (m)")
axes[0].legend()

axes[1].plot(t, y_true, label="y(t) exact")
axes[1].plot(t, y_pred, "--", label="y(t) PINN")
axes[1].set(title="y(t)", xlabel="t (s)", ylabel="y (m)")
axes[1].legend()

prediction_path = RUN_OUTPUT_DIR / "prediction.png"
figure.savefig(prediction_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"현재 실행 그래프 저장: {prediction_path}")


## 결과 해석 범위

- Validator는 학습 범위인 `0≤t<5 s`에서 정확도를 평가합니다.
- Inferencer가 만드는 `0≤t<8 s` 예측 중 `5–8 s`는 학습 범위를 벗어난 외삽 구간입니다.
- 물체는 수학적으로 약 `7.06 s`에 지면으로 돌아옵니다. 지면 충돌을 모델링하지 않았으므로 이후의 `y<0` 값은 운동방정식의 해를 연장한 결과입니다.
- 이 실습의 오차는 `v₀=40 m/s`, `θ=60°` 조건에서 ODE 해를 근사한 정확도를 나타냅니다. 다른 초기속도나 발사각은 별도의 입력·학습 구성이 필요합니다.


## 10. 다음 단계 — 고정된 초기조건의 PINN에서 새로운 소스항의 FNO로

| 구분 | 발사체 운동 PINN | Poisson FNO |
|---|---|---|
| 입력 | 시간 `t` | 격자 위 소스항 `f(x,y)` |
| 출력 | 고정된 초기조건의 궤적 `x(t), y(t)` | 소스항에 대응하는 해 `u(x,y)` |
| 학습 신호 | 초기조건 + ODE 잔차 | 여러 소스항과 해의 쌍 `(f,u)` |
| 학습 후 계산 | 같은 초기조건에서 새로운 시간의 위치 | 학습에서 보지 못한 소스항의 전체 해 |

**완료 체크리스트**

- [ ] 학습 프로세스가 종료 코드 0으로 끝났습니다.
- [ ] `validator.npz`에서 상대 L2 오차와 최대 절대 오차를 확인했습니다.
- [ ] 해석해와 PINN 예측 그래프를 비교했습니다.
- [ ] 학습 구간, 외삽 구간, 지면 충돌 이후 구간을 구분할 수 있습니다.

다음 [02_Poisson_FNO.ipynb](02_Poisson_FNO.ipynb)에서는 여러 소스항과 해의 예를 학습하고, 새로운 소스항에 대응하는 전체 해를 예측합니다. FNO 학습은 16:10 세션에서 강사 안내에 따라 시작합니다.


<details>
<summary><strong>선택 부록 · TensorBoard와 ParaView</strong></summary>

KISTI에서는 현재 실행의 기록만 보도록 `tensorboard --logdir <RUN_OUTPUT_DIR> --port 8889 --host 127.0.0.1`로 실행하고, 강사가 안내한 SSH 터널을 통해서만 접속합니다. 승인 없이 TensorBoard를 외부 인터페이스에 열지 않습니다. 로컬 Docker에서는 컨테이너 안의 TensorBoard를 `--host 0.0.0.0`으로 실행하되, 포트는 `-p 127.0.0.1:8889:8889`처럼 호스트의 loopback에만 연결합니다.

ParaView는 `RUN_OUTPUT_DIR` 아래의 VTK/VTP 파일을 내려받아 결과를 추가로 살펴볼 때 사용합니다. 이번 KSC 필수 과정은 노트북에서 Matplotlib 그래프를 확인하는 단계까지입니다.
</details>


---

## 참고 문헌과 라이선스

- [PhysicsNeMo-Sym 문서](https://docs.nvidia.com/deeplearning/physicsnemo/physicsnemo-sym/)
- [물리 정보 신경망(PINN) 논문](https://www.sciencedirect.com/science/article/pii/S0021999118307125)

Copyright © 2026 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). Existing file-level notices remain in effect.
